# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists("Capstone-Flyrank"):
    !git clone https://github.com/J-Aamir/Capstone-Flyrank.git
%cd Capstone-Flyrank

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df.trend_direction == "down").astype(int)
d = df[df.avg_position > 0].copy()
print(d.shape)


Cloning into 'Capstone-Flyrank'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 198 (delta 92), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 1.89 MiB | 11.00 MiB/s, done.
Resolving deltas: 100% (92/92), done.
/content/Capstone-Flyrank
(28795, 45)


## 1. Two paper findings + my methodology questions

**Finding chosen #1 — ML Appendix, "What Predicts Growth?"** (logistic regression, 71% holdout accuracy, 80/20 split). The label (growth vs. decline from 30d-vs-prev-30d impressions) is a genuinely observed outcome, not a defined rule, so that part is solid. **My methodology question:** the paper discloses an 80/20 split but doesn't say whether it's row-random or grouped by brand/client. This matters a lot in practice: in my own Week-5 audit, a row-random split let 31 of 32 clients leak into both train and test, inflating Precision@50 from an honest 0.74 to an artificially high 0.92. With 57 brands in this portfolio, the same risk applies here — if the split was row-random, the 71% accuracy may be partly measuring "the model recognized a brand it already saw," not "the model generalizes to a new brand's pages." This is a constructive question, not an accusation: the paper may already control for this and just not state it in the public version.

**Finding chosen #2 — Finding #2, "The Content Performance Curve"** (health score by content age, peaking at 61-90 days, dropping to a "decay cliff" by 271-365 days). **My methodology question:** the paper's own Methodology section states Health Score is a composite of impressions (30pts) + position (30pts) + CTR (20pts) + scroll depth (20pts) — i.e., it's partly built FROM visibility metrics. The paper is careful to flag this circularity risk for the ML feature-importance section ("health score is partly constructed from these inputs... read as model behavior, not causation"), but the same caution isn't explicitly repeated for the age-curve finding. Does the age-driven "decay cliff" partly reflect that older, declining pages naturally have lower position/impressions (direct health-score inputs), rather than age itself driving an independent decline? The paper's own correlation appendix shows content_age_days vs. health score at only r=0.0, which seems to support that age alone is weak and the curve is likely carried by the metrics health score is literally built from. Worth a one-line caveat in the public version.

In [2]:
# Supporting check: the paper's own correlation matrix shows content_age_days vs health_score ~ 0.0,
# while avg_position vs health_score is -0.6 (strong) -- consistent with the Section 1 question above.
print("From the paper's correlation appendix:")
print("  content_age_days vs health_score: r = 0.0 (near zero)")
print("  avg_position     vs health_score: r = -0.6 (strong)")
print("This supports the question: the age-driven decay curve may be carried by position/impressions,")
print("not by age itself -- since age barely correlates with health score directly.")


From the paper's correlation appendix:
  content_age_days vs health_score: r = 0.0 (near zero)
  avg_position     vs health_score: r = -0.6 (strong)
This supports the question: the age-driven decay curve may be carried by position/impressions,
not by age itself -- since age barely correlates with health score directly.


## 2. My model under an honest split (before/after)

Re-running my Week-5 Random Forest two ways on the same data: a naive **row-random** split (the dishonest "before") vs. a **client-grouped** split (the honest "after", same as Week 5). This is the exact issue I raised about the paper's own 80/20 split in Section 1, demonstrated on my own work first.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features = ['content_age_days','days_since_last_update','impressions_90d','avg_position',
            'ctr','word_count','search_volume']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive row-random split
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
    d[features].fillna(0), d['is_declining_label'], test_size=0.3, random_state=42)
rf_r = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced',
                               random_state=42, n_jobs=-1).fit(Xtr_r, ytr_r)
scores_r = rf_r.predict_proba(Xte_r)[:, 1]

print("BEFORE (row-random split):")
for k in (20, 50):
    print(f"  Precision@{k}: {precision_at_k(scores_r, yte_r.values, k):.3f}")

client_overlap = len(set(d.loc[Xtr_r.index, 'client_id']) & set(d.loc[Xte_r.index, 'client_id']))
print(f"  Clients leaking into both train and test: {client_overlap} / {d['client_id'].nunique()}")
print()

# AFTER: client-grouped split (honest, same design as Week 5)
clients = d['client_id'].unique()
train_clients, test_clients = train_test_split(clients, test_size=0.3, random_state=42)
train = d[d.client_id.isin(train_clients)]
test  = d[d.client_id.isin(test_clients)]
Xtr_g, Xte_g = train[features].fillna(0), test[features].fillna(0)
ytr_g, yte_g = train['is_declining_label'].values, test['is_declining_label'].values
rf_g = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced',
                               random_state=42, n_jobs=-1).fit(Xtr_g, ytr_g)
scores_g = rf_g.predict_proba(Xte_g)[:, 1]

print("AFTER (client-grouped split):")
for k in (20, 50):
    print(f"  Precision@{k}: {precision_at_k(scores_g, yte_g, k):.3f}")


BEFORE (row-random split):
  Precision@20: 0.900
  Precision@50: 0.920
  Clients leaking into both train and test: 31 / 31

AFTER (client-grouped split):
  Precision@20: 0.700
  Precision@50: 0.740


## 3. Leakage audit

The same hunt from Week 3, applied to my final Week-5/6 feature set.

In [4]:
final_features = ['content_age_days','days_since_last_update','impressions_90d','avg_position',
                   'ctr','word_count','search_volume']
label_source_cols = ['trend_direction', 'trend_pct']

overlap = set(final_features) & set(label_source_cols)
print("Final feature set:", final_features)
print("Label-source columns (never allowed as features):", label_source_cols)
print("Overlap (must be empty):", overlap)
assert len(overlap) == 0, "Leakage detected: a label-source column is in the feature set!"
print("PASS: no direct leakage from label-source columns.\n")

# Secondary check: correlation of each feature with the label, flagging anything suspiciously high
# (a near-1.0 correlation would suggest the feature is itself a disguised copy of the label)
corrs = d[final_features + ['is_declining_label']].corr()['is_declining_label'].drop('is_declining_label')
print("Feature correlation with label (flag anything near +-1.0):")
print(corrs.round(3).sort_values())
print()
print("No feature exceeds a moderate correlation, consistent with no hidden leakage.")


Final feature set: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'search_volume']
Label-source columns (never allowed as features): ['trend_direction', 'trend_pct']
Overlap (must be empty): set()
PASS: no direct leakage from label-source columns.

Feature correlation with label (flag anything near +-1.0):
content_age_days         -0.179
avg_position             -0.081
ctr                      -0.069
impressions_90d          -0.033
search_volume            -0.020
days_since_last_update    0.053
word_count                0.066
Name: is_declining_label, dtype: float64

No feature exceeds a moderate correlation, consistent with no hidden leakage.


## 4. Claim rewrite

My boldest sentence from Week 5 was: *"The Random Forest beat the Week-4 baseline at both K (Precision@20: 0.700 vs 0.550; Precision@50: 0.740 vs 0.580)."* That's already in careful numeric language, but it implies a single, settled result. Section 2 above shows the honest number depends heavily on split design. Rewritten safely:

In [5]:
rewrite = (
    "Observed: on a client-grouped holdout split, the Random Forest's Precision@50 (0.740) was "
    "directionally higher than the Week-4 rule-based baseline (0.580) on this dataset. This is a "
    "decision-support signal, not a guarantee of generalization -- the same model scored an "
    "inflated 0.920 under a naive row-random split where 31 of 32 clients leaked into both train "
    "and test, so the grouped number is the one I trust, and even it reflects performance on this "
    "specific sample of clients, not a universal claim about all content portfolios."
)
print(rewrite)


Observed: on a client-grouped holdout split, the Random Forest's Precision@50 (0.740) was directionally higher than the Week-4 rule-based baseline (0.580) on this dataset. This is a decision-support signal, not a guarantee of generalization -- the same model scored an inflated 0.920 under a naive row-random split where 31 of 32 clients leaked into both train and test, so the grouped number is the one I trust, and even it reflects performance on this specific sample of clients, not a universal claim about all content portfolios.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.